In [1]:
import pandas as pd

In [2]:
# Loading Data and Dropping Null Values

# data = pd.read_csv('../Data/Jan/202401-divvy-tripdata.csv')
# data = pd.read_csv('../Data/Feb/202402-divvy-tripdata.csv')
# data = pd.read_csv('../Data/March/202403-divvy-tripdata.csv')
# data = pd.read_csv('../Data/April/202404-divvy-tripdata.csv')
# data = pd.read_csv('../Data/May/202405-divvy-tripdata.csv')
# data = pd.read_csv('../Data/June/202406-divvy-tripdata.csv') 
# data = pd.read_csv('../Data/July/202407-divvy-tripdata.csv') 
# data = pd.read_csv('../Data/Aug/202408-divvy-tripdata.csv')
# data = pd.read_csv('../Data/Sept/202409-divvy-tripdata.csv')
# data = pd.read_csv('../Data/Oct/202410-divvy-tripdata.csv')
# data = pd.read_csv('../Data/Nov/202411-divvy-tripdata.csv')
# data = pd.read_csv('../Data/Dec/202412-divvy-tripdata.csv')


In [3]:
data = pd.read_csv('../Data/Divvy_2024_All_Months_Cleaned.csv')

In [4]:
data.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
# Convert started_at and ended_at to datetime
data['started_at'] = pd.to_datetime(data['started_at'], format='mixed').dt.floor('s')
data['ended_at']   = pd.to_datetime(data['ended_at'], format='mixed').dt.floor('s')


# Ensure latitude and longitude columns are float
lat_lng_cols = ['start_lat', 'start_lng', 'end_lat', 'end_lng']
data[lat_lng_cols] = data[lat_lng_cols].astype(float)

# Ensure station ID columns are string
station_id_cols = ['start_station_id', 'end_station_id']
data[station_id_cols] = data[station_id_cols].astype(str)

In [37]:
data.dropna(inplace=True)   
data.reset_index(drop=True, inplace=True)

In [38]:
# Taking a random sample of 10% of the data

sample = data.sample(frac=0.1, random_state=42)
sample.reset_index(drop=True, inplace=True) 
# print(sample.info())

print((sample['start_station_name' ].value_counts(normalize=True)*100).head(10))

start_station_name
Streeter Dr & Grand Ave               1.421035
DuSable Lake Shore Dr & Monroe St     1.005180
Michigan Ave & Oak St                 0.941020
Kingsbury St & Kinzie St              0.891117
DuSable Lake Shore Dr & North Blvd    0.834086
Clinton St & Washington Blvd          0.767549
Clark St & Elm St                     0.720023
Millennium Park                       0.705765
Theater on the Lake                   0.696260
Clinton St & Madison St               0.689131
Name: proportion, dtype: float64


In [39]:
# Encode 'rideable_type' and 'member_casual' as categorical codes

sample['rideable_type_code'] = sample['rideable_type'].astype('category').cat.codes
sample['member_casual_code'] = sample['member_casual'].astype('category').cat.codes

In [40]:
print(sample.columns)

Index(['ride_id', 'rideable_type', 'rideable_type_code', 'started_at',
       'ended_at', 'ride_length', 'month_name', 'day_of_week', 'hour_of_day',
       'start_station_name', 'start_station_id', 'start_lat', 'start_lng',
       'end_station_name', 'end_station_id', 'end_lat', 'end_lng',
       'member_casual', 'member_casual_code'],
      dtype='object')


In [41]:
sample['ride_length'] = (sample['ended_at'] - sample['started_at']).dt.total_seconds() / 60 
sample['hour_of_day'] = sample['started_at'].dt.strftime('%I %p')
sample['month_name'] = sample['started_at'].dt.strftime("%B")
sample['day_of_week'] = sample['started_at'].dt.day_name()

In [42]:
# Remove rides with negative or excessively long ride lengths
sample = sample[(sample['ride_length'] >= 0) & 
                                (sample['ride_length'] <= 1440)]

In [43]:
# Filter to Chicago area based on lat/lng bounds
sample = sample[(sample['start_lat'].between(41.6, 42.1)) &
                                (sample['start_lng'].between(-88.0, -87.5)) &
                                (sample['end_lat'].between(41.6, 42.1)) &
                                (sample['end_lng'].between(-88.0, -87.5))]  


In [44]:
# Encode 'rideable_type' and 'member_casual' as categorical codes
reordered_columns = ['ride_id',
 'rideable_type', 'rideable_type_code',
 'started_at', 'ended_at',
 'ride_length', 'month_name', 'day_of_week', 'hour_of_day',
 'start_station_name', 'start_station_id', 'start_lat', 'start_lng',
 'end_station_name', 'end_station_id', 'end_lat', 'end_lng',
 'member_casual', 'member_casual_code']

sample = sample[reordered_columns]

print(sample.head())

            ride_id  rideable_type  rideable_type_code          started_at  \
0  754A131D58BEA039   classic_bike                   0 2024-09-01 07:57:40   
1  2BA8F47F3C67892E   classic_bike                   0 2024-10-29 19:13:07   
2  C06EA1BC1E072633  electric_bike                   1 2024-10-11 10:45:14   
3  696A7CB094494171   classic_bike                   0 2024-11-26 09:16:55   
4  9F54EDCE55416897  electric_bike                   1 2024-10-03 10:29:16   

             ended_at  ride_length month_name day_of_week hour_of_day  \
0 2024-09-01 08:24:12    26.533333  September      Sunday       07 AM   
1 2024-10-29 22:01:29   168.366667    October     Tuesday       07 PM   
2 2024-10-11 10:48:52     3.633333    October      Friday       10 AM   
3 2024-11-26 09:23:30     6.583333   November     Tuesday       09 AM   
4 2024-10-03 10:33:52     4.600000    October    Thursday       10 AM   

                 start_station_name start_station_id  start_lat  start_lng  \
0         Cann

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420816 entries, 0 to 420815
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             420816 non-null  object        
 1   rideable_type       420816 non-null  object        
 2   rideable_type_code  420816 non-null  int64         
 3   started_at          420816 non-null  datetime64[ns]
 4   ended_at            420816 non-null  datetime64[ns]
 5   ride_length         420816 non-null  float64       
 6   month_name          420816 non-null  object        
 7   day_of_week         420816 non-null  object        
 8   hour_of_day         420816 non-null  object        
 9   start_station_name  420816 non-null  object        
 10  start_station_id    420816 non-null  object        
 11  start_lat           420816 non-null  float64       
 12  start_lng           420816 non-null  float64       
 13  end_station_name    420816 no

In [8]:
data.to_csv('../Data/Divvy_2024_All_Months_Cleaned.csv')

In [47]:
# Converting the cleaned sample to a CSV file

# sample.to_csv('../Data/Jan/Divvy_cleaned_Jan_2024.csv')
# sample.to_csv('../Data/Feb/Divvy_cleaned_Feb_2024.csv')
# sample.to_csv('../Data/March/Divvy_cleaned_March_2024.csv')
# sample.to_csv('../Data/April/Divvy_cleaned_April_2024.csv')
# sample.to_csv('../Data/May/Divvy_cleaned_May_2024.csv')
# sample.to_csv('../Data/June/Divvy_cleaned_June_2024.csv')
# sample.to_csv('../Data/July/Divvy_cleaned_July_2024.csv')
# sample.to_csv('../Data/Aug/Divvy_cleaned_Aug_2024.csv')
# sample.to_csv('../Data/Sept/Divvy_cleaned_Sept_2024.csv')
# sample.to_csv('../Data/Oct/Divvy_cleaned_Oct_2024.csv')
# sample.to_csv('../Data/Nov/Divvy_cleaned_Nov_2024.csv')
# sample.to_csv('../Data/Dec/Divvy_cleaned_Dec_2024.csv')